# AI Competitiveness Final Submission Pipeline

## Phase and Workflow Bridge
This notebook is the submission-ready, phase-mapped execution notebook for the MIE1624 project. It reproduces baseline scoring, calibration-linked impact structure, quarterly simulation outputs, KPI gap attribution, and final output artifacts using only local package files.

Phase alignment in this notebook:
- Phase 1: Baseline rebuild and validation
- Phase 2: Calibration library loading and checks
- Phase 3: Recommendation-to-KPI impact structure
- Phase 4: Quarterly Monte Carlo simulation across scenarios
- Phase 5: KPI gap reduction attribution
- Phase 6 link: outputs generated here feed downstream dashboard/report layers (outside this notebook)

Upstream dependency: none inside this package (raw workbook and packaged CSV inputs are local).
Downstream dependency: outputs/tables and outputs/figures generated here are reused by dashboard and reporting workflows.

## 1. Project Scope, Data Flow, and Reproducibility Notes

### Scope
- Geography/ranking universe: fixed 13-country baseline with Canada tracked against peers.
- Time granularity and horizon: quarterly, 16 quarters from 2026Q1 to 2029Q4.
- Scenario set: baseline, rec1_only, rec2_only, rec3_only, all_recommendations.
- Uncertainty reporting: p10 (10th percentile), median (50th percentile), p90 (90th percentile).

### End-to-end workflow map
raw workbook + canada gap csv -> baseline reconstruction and validation (Phase 1) -> calibration libraries (Phase 2) -> recommendation-to-KPI impact mapping (Phase 3) -> quarterly Monte Carlo simulation (Phase 4) -> KPI gap attribution (Phase 5) -> dashboard/report consumption (Phase 6).

### Why this structure matters
Each phase validates and emits artifacts consumed by later phases. Baseline integrity and calibration consistency are preconditions for interpretable simulation and attribution outputs.

### Reproducibility assumptions
- Run cells top-to-bottom once in a clean kernel.
- Use only packaged local inputs; no external retrieval.
- Preserve deterministic configuration where specified (for example random seed in simulation config).

In [ ]:
from pathlib import Path
import sys

# Bootstrap for local and Colab execution from notebook location
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_TABLES = PROJECT_ROOT / 'outputs' / 'tables'
OUTPUT_FIGURES = PROJECT_ROOT / 'outputs' / 'figures'
OUTPUT_TABLES.mkdir(parents=True, exist_ok=True)
OUTPUT_FIGURES.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)

## Local-Only Data Access Policy

Any web/API retrieval logic is intentionally excluded from execution. If retrieval code is needed for documentation, keep it commented out and read from package files instead.

In [ ]:
# Example of intentionally disabled external retrieval pattern (do not execute):
# import requests
# raw = requests.get('https://example.com/data.csv', timeout=30).text

# Local-file replacement used in this submission package:
import pandas as pd
local_mapping = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'impact_matrix_long.csv')
local_mapping.head(3)

## 2. Baseline Rebuild and Validation (Phase 1)

### Objective
Reconstruct country and KPI baseline scores from source workbook data and validate ranking/weight consistency before any policy scenario modeling.

### Why this phase is necessary
All downstream simulation and attribution depend on a stable baseline state. If baseline values or rank order are inconsistent, policy effects cannot be interpreted reliably.

### Inputs
- data/raw/AI_Competitiveness_Rankings_and_Weights (1).xlsx
  - required sheets: KPI Scores and Weights, Final Rankings
- data/raw/canada_combined_kpi_weights_gaps.csv

### Core aggregation formulas
- weighted_kpi_score = kpi_score * kpi_weight
- subpillar_score = sum(weighted_kpi_score within subpillar)
- weighted_subpillar_score = subpillar_score * subpillar_weight
- composite_score = sum(weighted_subpillar_score across subpillars)
- country rank = descending composite_score with deterministic tie handling

### Assumptions
- Workbook scores are treated as scoring-ready values in this baseline module.
- Country universe is fixed to 13 countries.
- Expected workbook schema/labels are fixed and validated.

### Validation / QA focus
- required sheet presence and mapping checks
- country count and KPI count checks
- AHP L1/L2 weight consistency checks
- Canada target and rank checks
- full rank-order comparison against workbook reference
- cross-file consistency with Canada gap CSV

### Outputs written
- outputs/tables/baseline_country_scores.csv
- outputs/tables/baseline_kpi_scores.csv
- outputs/tables/validation_report.md

### Implementation note
No artifact named baseline_master_table is generated; baseline_country_scores.csv and baseline_kpi_scores.csv are the canonical baseline outputs.

In [ ]:
from src.scoring.baseline_reproduction import reproduce_baseline

baseline_result = reproduce_baseline(
    workbook_path=PROJECT_ROOT / 'data' / 'raw' / 'AI_Competitiveness_Rankings_and_Weights (1).xlsx',
    canada_gap_csv_path=PROJECT_ROOT / 'data' / 'raw' / 'canada_combined_kpi_weights_gaps.csv',
    output_dir=OUTPUT_TABLES,
)

baseline_result.country_scores[['rank', 'country', 'composite_score']].head(13)

In [ ]:
baseline_result.validation_checks[['section', 'check', 'passed']].head(20)

## 3. Calibration Input Loading (Phase 2)

### Objective
Load and verify calibration libraries that encode policy effect size priors, lag priors, competitor drift priors, and implementation timing assumptions.

### Why this phase is necessary
The simulation engine requires externally calibrated parameters not contained in baseline tables.

### Inputs and intended role
- calibration_effects.csv: low/base/high effect anchors plus confidence labels by lever
- calibration_lags.csv: low/base/high lag quarter anchors and ramp archetypes
- competitor_drift.csv: cluster-level KPI drift priors for non-Canada trajectories
- implementation_schedule.csv: start_quarter, full_effect_quarter, rollout_pattern per lever

### Assumptions and rationale
- Low/base/high anchors capture uncertainty bounds where direct AI-only estimates are limited.
- Confidence labels are explicit uncertainty signals used downstream in effect scaling.
- Some schedule rows include inferred starter assumptions documented in source notes.

### Validation / QA focus
- required-column validation for each calibration table
- lever-id uniqueness and cross-file key integrity checks in downstream loaders

### Downstream link
These parameter libraries are consumed by Phase 3 impact mapping and Phase 4 simulation logic.

In [ ]:
cal_effects = pd.read_csv(PROJECT_ROOT / 'data' / 'calibration' / 'calibration_effects.csv')
cal_lags = pd.read_csv(PROJECT_ROOT / 'data' / 'calibration' / 'calibration_lags.csv')
cal_drift = pd.read_csv(PROJECT_ROOT / 'data' / 'calibration' / 'competitor_drift.csv')
cal_schedule = pd.read_csv(PROJECT_ROOT / 'data' / 'calibration' / 'implementation_schedule.csv')

{
    'effects_rows': len(cal_effects),
    'lags_rows': len(cal_lags),
    'drift_rows': len(cal_drift),
    'schedule_rows': len(cal_schedule),
}

## 4. Recommendation-to-KPI Impact Structure (Phase 3)

### Objective
Translate recommendation/lever design into simulation-ready KPI coefficients and lag metadata.

### Why this phase is necessary
This phase bridges policy design language and quantitative state-transition simulation inputs.

### Inputs
- outputs/tables/baseline_kpi_scores.csv
- data/processed/impact_matrix_long.csv
- data/calibration/calibration_effects.csv
- data/calibration/calibration_lags.csv

### Core method logic
- Canonicalize recommendation and lever identifiers.
- Convert effect_strength buckets to base coefficients (0:0.0000, 1:0.0100, 2:0.0250, 3:0.0400).
- Adjust coefficients using effect anchors, confidence factors, and direction.
- Map lag classes to lag-quarter selections from calibration priors.
- Emit long metadata and wide lever x KPI matrix artifacts.

### Assumptions
- direct/indirect/spillover annotations are structural priors, not causal estimates.
- Archetype overrides are rule-based for lever/KPI contexts where exact calibration rows are unavailable.

### Validation / QA focus
- baseline KPI existence checks for mapped KPI names
- duplicate (lever, KPI) mapping checks
- required-column checks and long-vs-wide consistency checks

### Outputs used downstream
- outputs/tables/impact_matrix_long.csv
- outputs/tables/impact_matrix.csv
These are required by Phase 4 simulation.

In [ ]:
from src.scoring.policy_impact_matrix import build_policy_impact_artifacts

impact_result = build_policy_impact_artifacts(
    baseline_kpi_path=OUTPUT_TABLES / 'baseline_kpi_scores.csv',
    recommendation_mapping_path=PROJECT_ROOT / 'data' / 'processed' / 'impact_matrix_long.csv',
    calibration_effects_path=PROJECT_ROOT / 'data' / 'calibration' / 'calibration_effects.csv',
    calibration_lags_path=PROJECT_ROOT / 'data' / 'calibration' / 'calibration_lags.csv',
    output_dir=OUTPUT_TABLES,
)

impact_result.metadata[['recommendation_id', 'lever_id', 'KPI', 'numeric_coefficient', 'lag_quarters']].head(12)

## 5. Simulation Engine and Quarterly Scenario Generation (Phase 4)

### Objective
Simulate quarterly KPI/composite/rank trajectories across five scenarios under calibrated policy effects, rollout timing, lag dynamics, and competitor drift.

### Horizon and scenario design
- Horizon: 16 quarters from 2026Q1 to 2029Q4
- Scenarios: baseline, rec1_only, rec2_only, rec3_only, all_recommendations

### Inputs
- baseline_country_scores.csv and baseline_kpi_scores.csv
- impact_matrix.csv and impact_matrix_long.csv
- implementation_schedule.csv
- calibration_effects.csv, calibration_lags.csv, competitor_drift.csv

### Core algorithmic logic
- Canada path: apply policy shifts by active scenario, rollout profile, and lag response.
- Non-Canada path: apply competitor drift by calibrated cluster/KPI priors.
- Recompute sub-pillar, pillar, composite score, and rank each quarter.

### Representative transition/update formulas
- non-Canada drift update: state_next = state + drift_rate * ((100 - state) / 100)
- Canada policy shift (conceptual): canada_state = baseline_state + policy_shift (bounded to score limits)

### Monte Carlo and uncertainty
- Draw-level uncertainty sampling around low/base/high settings (triangular-style bounded sampling in implementation).
- Reported quantiles: p10 (10th percentile), median (50th percentile), p90 (90th percentile).
- Median is used as the main presentation path for robustness against optimistic tail draws.

### Validation / QA focus
- required-column and schema checks on all simulation inputs
- baseline recomputation consistency checks
- impact metadata/matrix consistency checks
- cross-file lever-id integrity checks

### Outputs written
- outputs/tables/simulation_summary.csv
- outputs/tables/country_quarter_scores.csv
- outputs/tables/canada_kpi_trajectories.csv
- outputs/tables/canada_rank_trajectory.csv
- outputs/tables/scenario_comparison.csv

In [ ]:
from src.simulation.state_transition import StateTransitionConfig, run_quarterly_state_transition

sim_config = StateTransitionConfig(start_quarter='2026Q1', quarters=16, draws=10000, random_seed=42)
simulation_result = run_quarterly_state_transition(
    config=sim_config,
    baseline_country_path=OUTPUT_TABLES / 'baseline_country_scores.csv',
    baseline_kpi_path=OUTPUT_TABLES / 'baseline_kpi_scores.csv',
    impact_matrix_path=OUTPUT_TABLES / 'impact_matrix.csv',
    impact_metadata_path=OUTPUT_TABLES / 'impact_matrix_long.csv',
    implementation_schedule_path=PROJECT_ROOT / 'data' / 'calibration' / 'implementation_schedule.csv',
    calibration_effects_path=PROJECT_ROOT / 'data' / 'calibration' / 'calibration_effects.csv',
    calibration_lags_path=PROJECT_ROOT / 'data' / 'calibration' / 'calibration_lags.csv',
    competitor_drift_path=PROJECT_ROOT / 'data' / 'calibration' / 'competitor_drift.csv',
    output_dir=OUTPUT_TABLES,
)

simulation_result.scenario_comparison

## 6. Monte Carlo Outputs and Headline Selection

### Why this section exists
Inspect distribution-aware outputs while documenting why median paths are used for most technical communication and dashboard surfaces.

### Quantile definitions
- p10: conservative lower-tail outcome across draws
- median: central tendency used as headline path
- p90: optimistic upper-tail outcome across draws

### Interpretation note
Distribution bands are retained in outputs for technical review, but median series are foregrounded in summary tables and dashboard-facing layers for interpretability and stability.

In [ ]:
simulation_result.canada_rank_trajectory[['scenario', 'quarter_index', 'rank_median', 'composite_score_median']].head(20)

## 7. KPI Gap Attribution (Phase 5)

### Objective
Quantify final-horizon KPI gap reduction for each stand-alone recommendation scenario versus benchmark references.

### Why attribution is after simulation
Simulation generates trajectories; attribution then evaluates end-state benchmark gap closure using those simulated final-quarter KPI values.

### Benchmark definitions
- peer_best: best KPI score among Canada peer-cluster comparators
- global_best: best KPI score across all baseline countries

### Final-quarter extraction logic
Use the terminal quarter in canada_kpi_trajectories.csv for scenario KPI comparison against benchmark scores.

### Attribution formulas
- baseline_gap = max(benchmark_score - canada_baseline_score, 0)
- post_policy_gap = max(benchmark_score - canada_postpolicy_score, 0)
- gap_reduction_absolute = baseline_gap - post_policy_gap
- gap_reduction_percent = gap_reduction_absolute / baseline_gap (if baseline_gap > 0 else 0)

### Interpretation constraint
Stand-alone recommendation effects are non-additive. Combined scenario effects should not be interpreted as arithmetic sums of stand-alone reductions.

### Validation / QA focus
- required scenario presence checks
- duplicate (scenario, KPI) final-quarter checks
- benchmark alignment checks

### Outputs written
- outputs/tables/kpi_gap_reduction_by_recommendation.csv
- outputs/tables/kpi_gap_reduction_peer_best.csv
- outputs/tables/kpi_gap_reduction_global_best.csv
- outputs/figures/recommendation_kpi_heatmap.png
- outputs/figures/rec1_gap_reduction.png
- outputs/figures/rec2_gap_reduction.png
- outputs/figures/rec3_gap_reduction.png
- outputs/figures/combined_gap_reduction.png

In [ ]:
from src.simulation.recommendation_attribution import export_recommendation_attribution

attr_paths = export_recommendation_attribution(
    baseline_kpi_path=OUTPUT_TABLES / 'baseline_kpi_scores.csv',
    canada_trajectory_path=OUTPUT_TABLES / 'canada_kpi_trajectories.csv',
    tables_output_dir=OUTPUT_TABLES,
    figures_output_dir=OUTPUT_FIGURES,
    canada_country='Canada',
)

{k: str(v) for k, v in attr_paths.items()}

## 8. Final Output Tables and Figures

Purpose: list and validate generated reporting-ready artifacts used by downstream dashboard/report workflows.

In [ ]:
generated_tables = sorted(p.name for p in OUTPUT_TABLES.glob('*.csv'))
generated_figures = sorted(p.name for p in OUTPUT_FIGURES.glob('*.png'))
{'tables': generated_tables, 'figures': generated_figures}

## 9. Reproducibility, Assumptions, and Phase-6 Integration Notes

### Execution notes
- This notebook is implementation-grounded and local-data-only.
- Run top-to-bottom to regenerate all core tables/figures used downstream.
- Keep paths and filenames unchanged for deterministic package behavior.

### Cross-phase assumptions recap
- Baseline workbook and packaged calibration files are canonical inputs.
- Calibration includes proxy-based anchors where direct evidence is limited.
- Attribution is final-quarter and scenario-stand-alone by design.

### Phase 6 (dashboard/report) linkage
Downstream dashboard loaders consume baseline, simulation, attribution, impact, calibration, and tracking tables. Dashboard layers typically foreground median paths and selected KPI subsets for readability while full distribution outputs remain in backend exports.

### Known interpretation limits
- Not all backend KPIs are front-facing in dashboard views.
- Benchmark choice (peer_best vs global_best) changes attribution interpretation.
- Scenario interaction effects imply non-additivity across stand-alone recommendation outputs.

### Package note
This submission package intentionally excludes exploratory and non-essential draft assets; dependency traces are documented in checks/notebook_dependency_map.md and checks/submission_manifest.csv.

## 10. Final Phase Summary and Downstream Bridge

### What this notebook produced
- Validated baseline tables
- Calibrated impact metadata and matrix
- Quarterly simulation outputs across five scenarios
- KPI gap attribution tables and figures
- Consolidated output inventories for reproducibility checks

### Output locations
- outputs/tables/*.csv
- outputs/figures/*.png

### How outputs are used next
- Dashboard assembly uses baseline/simulation/attribution outputs to build executive overview and timeline views.
- Reporting and technical review use these files for traceability, QA, and reproducibility verification.